# AnomaForge — Stage 1: Explore the MVTec AD `bottle` dataset

**Goal:** Confirm our dataset paths, inspect normal and defective sample images, view a real matching ground-truth defect mask, and check the planned **224 × 224** model-input shape.

This notebook contains **no neural-network models or training**. Real defective images and masks are inspected for understanding and will be reserved for final evaluation in the project.

## 1. Mount Google Drive

Upload/extract the official MVTec AD `bottle` folder to your Drive before running the remaining cells. Only change the path in the next cell if your folder is somewhere else.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Point the notebook to the extracted `bottle` folder

The `BOTTLE_DIR` folder should directly contain the `train/`, `test/`, and `ground_truth/` directories. Edit the path if necessary.

In [ ]:
from pathlib import Path

# EDIT THIS if you extracted the bottle folder somewhere else.
BOTTLE_DIR = Path('/content/drive/MyDrive/AnomaForge/data/mvtec_ad/bottle')

required = ['train/good', 'test/good', 'ground_truth']
missing = [x for x in required if not (BOTTLE_DIR / x).is_dir()]
if missing:
    raise FileNotFoundError(
        f"Could not find the expected folders under {BOTTLE_DIR}.\n"
        f"Missing: {missing}\n"
        "Extract the official 'bottle' category into Drive and update BOTTLE_DIR."
    )
print('Dataset folder found:', BOTTLE_DIR)

## 3. Count images and inspect the available labels

The training split should consist of normal images. The test split contains normal images and actual defects. We do not use real defect masks to train our models.

In [ ]:
def image_files(folder):
    return sorted(p for p in folder.glob('*.png') if p.is_file())

print('Training categories:')
for folder in sorted((BOTTLE_DIR / 'train').iterdir()):
    if folder.is_dir():
        print(f'  {folder.name:18s} : {len(image_files(folder))} images')

print('\nTest categories:')
for folder in sorted((BOTTLE_DIR / 'test').iterdir()):
    if folder.is_dir():
        print(f'  {folder.name:18s} : {len(image_files(folder))} images')

print('\nGround-truth mask categories:')
for folder in sorted((BOTTLE_DIR / 'ground_truth').iterdir()):
    if folder.is_dir():
        print(f'  {folder.name:18s} : {len(image_files(folder))} masks')

## 4. Visualize one normal image and one real defective image with its matching mask

The official MVTec AD naming convention pairs a defective image such as `002.png` with a mask named `002_mask.png` inside the matching `ground_truth/<defect_type>/` folder. This is a **dataset-inspection step**, not model training.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

normal_paths = image_files(BOTTLE_DIR / 'train' / 'good')
assert normal_paths, 'No normal training images were found.'
normal_path = normal_paths[0]

defect_types = [p.name for p in (BOTTLE_DIR / 'test').iterdir()
                if p.is_dir() and p.name != 'good' and image_files(p)]
assert defect_types, 'No defective test images were found.'
defect_type = sorted(defect_types)[0]
defect_path = image_files(BOTTLE_DIR / 'test' / defect_type)[0]
mask_path = BOTTLE_DIR / 'ground_truth' / defect_type / f'{defect_path.stem}_mask.png'
if not mask_path.is_file():
    raise FileNotFoundError(f'Expected matching defect mask was not found: {mask_path}')

normal = Image.open(normal_path).convert('RGB')
defective = Image.open(defect_path).convert('RGB')
mask = Image.open(mask_path).convert('L')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(normal);    axes[0].set_title('Normal (training)')
axes[1].imshow(defective); axes[1].set_title(f'Defective (test): {defect_type}')
axes[2].imshow(mask, cmap='gray', vmin=0, vmax=255); axes[2].set_title('Matching ground-truth mask')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

print('Normal image:', normal_path.name, normal.size)
print('Defective image:', defect_path.name, defective.size)
print('Mask:', mask_path.name, mask.size)

## 5. Check the planned neural-network input size

Our later DINOv3 and Student CNN stages will use **224 × 224 RGB** images. Here we check only shape and pixel range; the teacher's prescribed normalization will be applied when we actually load it.

In [ ]:
import numpy as np
import torch

resized = normal.resize((224, 224), Image.Resampling.BILINEAR)
array = np.asarray(resized, dtype=np.float32) / 255.0
tensor = torch.from_numpy(array).permute(2, 0, 1).contiguous()

print('Prepared input tensor shape:', tuple(tensor.shape))
print('Pixel range:', float(tensor.min()), 'to', float(tensor.max()))
assert tuple(tensor.shape) == (3, 224, 224)
print('Stage 1 tensor check passed.')

## Done: Stage 1 completion checklist

- [ ] `bottle/train/good/` loaded from Drive.
- [ ] Test categories and image counts were displayed.
- [ ] Normal image, defective image and matching mask were shown.
- [ ] Input tensor shape `(3, 224, 224)` passed.
- [ ] Notebook committed to the public GitHub repository **without** uploading datasets or credentials.

**Next stage:** Load the frozen DINOv3 teacher and check its spatial feature output. Do not build NN2 or NN3 yet.